# T001 - Diagnose the timestamp split failure

Kaggle version 1 returned None from the datetime quantile, producing empty partitions. No model or submission resulted. See run.log for the actual failure. The example below is synthetic, needs no competition data, and does not train or submit anything. The old quantile may work locally; its behavior is environment-dependent.


In [ ]:
from datetime import datetime, timedelta
import polars as pl

frame = pl.DataFrame({
    '_time': [datetime(2026, 10, 25) + timedelta(minutes=i // 2)
              for i in range(20)],
}).sort('_time')
print('Polars version:', pl.__version__)
print('Old quantile:', frame['_time'].quantile(0.8, interpolation='nearest'))
cutoff = frame['_time'][int(frame.height * 0.8)]
train = frame.filter(pl.col('_time') < cutoff)
valid = frame.filter(pl.col('_time') >= cutoff)
assert train.height == 16 and valid.height == 4
assert train['_time'].max() < valid['_time'].min()
print('Fixed cutoff:', cutoff, 'train:', train.height, 'valid:', valid.height)
